# 文本分类实例

## Step1 导入相关包

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## Step2 加载数据

In [2]:
import pandas as pd

# 数据集是 https://github.com/SophonPlus/ChineseNlpCorpus
data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [3]:
# 去掉空行
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [4]:
data.iloc[0]

,0
label,1
review,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."


In [5]:
data.iloc[0]["review"]

'距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.'

In [6]:
data.iloc[0]["label"]

np.int64(1)

## Step3 创建 Dataset

In [7]:
from torch.utils.data import Dataset


class MyDataset(Dataset):

    def __init__(self) -> None:
        super().__init__()
        self.data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
        self.data = self.data.dropna()

    def __getitem__(self, index):
        return self.data.iloc[index]["review"], self.data.iloc[index]["label"]

    def __len__(self):
        return len(self.data)

In [8]:
dataset = MyDataset()
for i in range(5):
    print(dataset[i])

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', np.int64(1))
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', np.int64(1))
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', np.int64(1))
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', np.int64(1))
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', np.int64(1))


## Step4 划分数据集

In [9]:
from torch.utils.data import random_split


trainset, validset = random_split(dataset, lengths=[0.9, 0.1])
len(trainset), len(validset)

(6989, 776)

In [10]:
for i in range(10):
    print(trainset[i])
    print(trainset[i][0])
    print(trainset[i][1])
    print()

('标准的商务酒店，最大的优点就是位置好。房间比较干净，设施还行，但有些细节还需要改进。楼层服务员很不错（上次住的6楼，在这里表扬一下这位服务员），收拾房间，做清洁很及时，认真，走的时候有东西忘了，追了我好长一段距离告诉我。最近浦东大道修路，附件象个打工地。', np.int64(1))
标准的商务酒店，最大的优点就是位置好。房间比较干净，设施还行，但有些细节还需要改进。楼层服务员很不错（上次住的6楼，在这里表扬一下这位服务员），收拾房间，做清洁很及时，认真，走的时候有东西忘了，追了我好长一段距离告诉我。最近浦东大道修路，附件象个打工地。
1

('刚才提到的如家快捷酒店，应该是如家深圳火车站店，在渔民村小区内。', np.int64(1))
刚才提到的如家快捷酒店，应该是如家深圳火车站店，在渔民村小区内。
1

('在携程上定了豪华大床房，要求A区6楼的房间，可惜到了的时候，由于都满，只有3层的，虽然视野不够开阔，还好也能看到湖，景色很美。服务也很好，服务员都会主动问好。打电话要风油精也很快送到。中餐厅太小，而且很吵，还要自己跑去点菜，鱼头汤味道不错，其他很一般。花园风景非常漂亮，游泳池很晚都有人在游泳，可惜这次时间不够，只在室内的游了游。网上介绍有烧烤，结果打电话询问说根本没有。晕。总的来说还是非常满意的，男朋友跟我一起回国三次，出去旅游了三次，这次是最满意的。特别是看到孔雀，兴奋不已。拍了很多照片，说下次还想来。', np.int64(1))
在携程上定了豪华大床房，要求A区6楼的房间，可惜到了的时候，由于都满，只有3层的，虽然视野不够开阔，还好也能看到湖，景色很美。服务也很好，服务员都会主动问好。打电话要风油精也很快送到。中餐厅太小，而且很吵，还要自己跑去点菜，鱼头汤味道不错，其他很一般。花园风景非常漂亮，游泳池很晚都有人在游泳，可惜这次时间不够，只在室内的游了游。网上介绍有烧烤，结果打电话询问说根本没有。晕。总的来说还是非常满意的，男朋友跟我一起回国三次，出去旅游了三次，这次是最满意的。特别是看到孔雀，兴奋不已。拍了很多照片，说下次还想来。
1

('总体来说还是不错的，就是酒店大堂设在商场里面给人感觉并不是很好，其他条件都尚可', np.int64(1))
总体来说还是不错的，就是酒店大堂设在商场里面给人感觉并不是很好，其他条件都尚可
1

('厦门首选

In [11]:
for i in range(10):
    print(validset[i])
    print(validset[i][0])
    print(validset[i][1])
    print()

('酒店的地理位置非常棒,住的高级商务间.感觉房间非常小.门童和服务生都非常热情.还有免费的水果.地毯和装修有些陈旧~', np.int64(1))
酒店的地理位置非常棒,住的高级商务间.感觉房间非常小.门童和服务生都非常热情.还有免费的水果.地毯和装修有些陈旧~
1

('同时订的两套房间，入住时间不同，最后拿到的房间也不同，很奇怪今晚酒店的房间怎么老是这样，当然都是升级到更高层次的房间。', np.int64(1))
同时订的两套房间，入住时间不同，最后拿到的房间也不同，很奇怪今晚酒店的房间怎么老是这样，当然都是升级到更高层次的房间。
1

('当地最好的，在全国算性价比不错的五星级酒店', np.int64(1))
当地最好的，在全国算性价比不错的五星级酒店
1

('很不错的酒店。虽然离地铁站有一定距离，可是还是挺方便的。房间的床很大很舒服。但是相对的，别的空间就缩小了。。。不过我还是很喜欢那张床。。。早上服务员的声音有些响希望可以改进下', np.int64(1))
很不错的酒店。虽然离地铁站有一定距离，可是还是挺方便的。房间的床很大很舒服。但是相对的，别的空间就缩小了。。。不过我还是很喜欢那张床。。。早上服务员的声音有些响希望可以改进下
1

('酒店前台的态度极其恶劣，我预定的是12月14日晚的双人间，携程预定是到晚上9点，由于行程改变，到达时间会推后，因为该酒店当天有一个全省会议，考虑到房间可能会紧张，所以在当晚8点前打电话到酒店前台，告知我会晚到，房间请保留，前台的答复是可以，当晚我9点30分到达酒店，前台告诉我过了预定的9点钟，房间已经取消，并且也没有房间了，还拿出携程的预定传真给我看，我说我已经事先电话告知会晚到并且已得到你们的承诺，现在房间取消竟然认为是我的责任，吵了半天来了一个大堂经理，上来又是同样的一堆废话，最后说只能给我一个大床间加一张床，没办法只能接受，对此事件非常遗憾，同时对于携程与这种酒店签订协议也很失望！', np.int64(0))
酒店前台的态度极其恶劣，我预定的是12月14日晚的双人间，携程预定是到晚上9点，由于行程改变，到达时间会推后，因为该酒店当天有一个全省会议，考虑到房间可能会紧张，所以在当晚8点前打电话到酒店前台，告知我会晚到，房间请保留，前台的答复是可以，当晚我9点30分到达酒店，前台告诉我过了预定的9点钟，房间

## Step5 创建 Dataloader

In [12]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")


def collate_func(batch):
    # print("batch:", batch)

    texts, labels = [], []
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    # print("texts:", texts)
    # print("labels:", labels)

    inputs = tokenizer(
        texts,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    # print(inputs.keys())
    inputs["labels"] = torch.tensor(labels)
    # print(inputs.keys())

    return inputs


"""
batch: [
    ('房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！', np.int64(1)),
    ('酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。', np.int64(1)),
    ('入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚度不够5星的。家具大都是原色的，地毯也是颜色的，不亮丽。电视陈旧，遥控器不灵了。比较好玩的是房间里的电水壶的大个头，好像还是进口的。还有早餐分中西常规、粤式(我倾向于这个)和日式的3个厅，不过只能选择其中一个，到10点结束。走廊是分片的感应灯。有夜床服务、午茶小饼干1块、枕头上会放一张别着小鲜花的印有诗句的卡纸，赞！洁具质量不好，除了肥皂。无烟楼层有客人在走廊和房间(房门洞开)抽烟，无人制止。缺少报纸供应。盥洗室放有女孩子可应急的发筋，细心，再赞一个。套房的景色很不错，离海倒是确实近，就在背后。稍微偏离一点商业区，不过不远，打车到五四广场3公里不到。补充点评2008年3月6日：差点忘了，我打车的司机径直把握开到东楼，拉门的先生还给我一张记录有车牌号的纸片，赞一下。另外，青岛的酒店服务普遍不错。不过饭店的服务员可能没听说过“菊花普洱”。', np.int64(1))
]

texts: [
    '房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！',
    '酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。',
    '入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚度不够5星的。家具大都是原色的，地毯也是颜色的，不亮丽。电视陈旧，遥控器不灵了。比较好玩的是房间里的电水壶的大个头，好像还是进口的。还有早餐分中西常规、粤式(我倾向于这个)和日式的3个厅，不过只能选择其中一个，到10点结束。走廊是分片的感应灯。有夜床服务、午茶小饼干1块、枕头上会放一张别着小鲜花的印有诗句的卡纸，赞！洁具质量不好，除了肥皂。无烟楼层有客人在走廊和房间(房门洞开)抽烟，无人制止。缺少报纸供应。盥洗室放有女孩子可应急的发筋，细心，再赞一个。套房的景色很不错，离海倒是确实近，就在背后。稍微偏离一点商业区，不过不远，打车到五四广场3公里不到。补充点评2008年3月6日：差点忘了，我打车的司机径直把握开到东楼，拉门的先生还给我一张记录有车牌号的纸片，赞一下。另外，青岛的酒店服务普遍不错。不过饭店的服务员可能没听说过“菊花普洱”。'
]

labels: [np.int64(1), np.int64(1), np.int64(1)]
"""

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


"\nbatch: [\n    ('房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！', np.int64(1)),\n    ('酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。', np.int64(1)),\n    ('入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚度不够5星的。家具大都是原色的，地毯也是颜色的，不亮丽。电视陈旧，遥控器不灵了。比较好玩的是房间里的电水壶的大个头，好像还是进口的。还有早餐分中西常规、粤式(我倾向于这个)和日式的3个厅，不过只能选择其中一个，到10点结束。走廊是分片的感应灯。有夜床服务、午茶小饼干1块、枕头上会放一张别着小鲜花的印有诗句的卡纸，赞！洁具质量不好，除了肥皂。无烟楼层有客人在走廊和房间(房门洞开)抽烟，无人制止。缺少报纸供应。盥洗室放有女孩子可应急的发筋，细心，再赞一个。套房的景色很不错，离海倒是确实近，就在背后。稍微偏离一点商业区，不过不远，打车到五四广场3公里不到。补充点评2008年3月6日：差点忘了，我打车的司机径直把握开到东楼，拉门的先生还给我一张记录有车牌号的纸片，赞一下。另外，青岛的酒店服务普遍不错。不过饭店的服务员可能没听说过“菊花普洱”。', np.int64(1))\n]\n\ntexts: [\n    '房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！',\n    '酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。',\n    '入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚

In [13]:
# 调试
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=3, shuffle=False, collate_fn=collate_func)

In [14]:
next(enumerate(trainloader))

(0,
 {'input_ids': tensor([[ 101, 3403, 1114, 4638, 1555, 1218, 6983, 2421, 8024, 3297, 1920, 4638,
           831, 4157, 2218, 3221,  855, 5390, 1962,  511, 2791, 7313, 3683, 6772,
          2397, 1112, 8024, 6392, 3177, 6820, 6121, 8024,  852, 3300,  763, 5301,
          5688, 6820, 7444, 6206, 3121, 6822,  511, 3517, 2231, 3302, 1218, 1447,
          2523,  679, 7231, 8020,  677, 3613,  857, 4638,  127, 3517, 8024, 1762,
          6821, 7027, 6134, 2813,  671,  678, 6821,  855, 3302, 1218, 1447, 8021,
          8024, 3119, 2896, 2791, 7313, 8024,  976, 3926, 3815, 2523, 1350, 3198,
          8024, 6371, 4696, 8024, 6624, 4638, 3198,  952, 3300,  691, 6205, 2563,
           749, 8024, 6841,  749, 2769, 1962, 7270,  671, 3667, 6655, 4895, 1440,
          6401, 2769,  511, 3297, 6818, 3855,  691, 1920, 6887,  934, 6662, 8024,
          7353,  816, 6496,  702, 2802, 2339, 1765,  102],
         [ 101, 1157, 2798, 2990, 1168, 4638, 1963, 2157, 2571, 2949, 6983, 2421,
          8024, 2418,

In [15]:
len(trainloader)

2330

In [16]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=32, shuffle=True, collate_fn=collate_func)
validloader = DataLoader(
    validset, batch_size=64, shuffle=False, collate_fn=collate_func
)

## Step6 创建模型及优化器

In [17]:
from torch.optim import Adam

model = AutoModelForSequenceClassification.from_pretrained("hfl/rbt3")

if torch.cuda.is_available():
    model = model.cuda()

model.device

pytorch_model.bin:   0%|          | 0.00/156M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


device(type='cuda', index=0)

In [18]:
optimizer = Adam(model.parameters(), lr=2e-5)

## Step7 训练与验证

In [19]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            print(f"evaluate pred: {pred.long()}, labels: {batch["labels"].long()}")
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    return acc_num / len(validset)


def train(epoch=3, log_step=10):
    global_step = 0

    for ep in range(epoch):
        model.train()
        # 每个 epoch 训练全部数据
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}

            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()

            if global_step % log_step == 0:
                print(
                    f"ep: {ep}, global_step: {global_step}, loss: {output.loss.item()}"
                )

            global_step += 1

        acc = evaluate()
        print(f"ep: {ep}, acc: {acc}")

## Step8 模型训练

In [20]:
train()

ep: 0, global_step: 0, loss: 0.7235499024391174
ep: 0, global_step: 10, loss: 0.5532006621360779
ep: 0, global_step: 20, loss: 0.48455810546875
ep: 0, global_step: 30, loss: 0.5498020648956299
ep: 0, global_step: 40, loss: 0.5204046368598938
ep: 0, global_step: 50, loss: 0.3223996162414551
ep: 0, global_step: 60, loss: 0.3804394006729126
ep: 0, global_step: 70, loss: 0.47620418667793274
ep: 0, global_step: 80, loss: 0.566184401512146
ep: 0, global_step: 90, loss: 0.38300031423568726
ep: 0, global_step: 100, loss: 0.42562317848205566
ep: 0, global_step: 110, loss: 0.2897989749908447
ep: 0, global_step: 120, loss: 0.47560814023017883
ep: 0, global_step: 130, loss: 0.1981586217880249
ep: 0, global_step: 140, loss: 0.24412864446640015
ep: 0, global_step: 150, loss: 0.3145613670349121
ep: 0, global_step: 160, loss: 0.38205191493034363
ep: 0, global_step: 170, loss: 0.11664622277021408
ep: 0, global_step: 180, loss: 0.26956692337989807
ep: 0, global_step: 190, loss: 0.1755605936050415
ep: 0,

## Step9 模型预测

In [21]:
sen = "我觉得这家酒店不错，饭很好吃！"
id2_label = {0: "差评！", 1: "好评！"}
model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt")
    inputs = {k: v.cuda() for k, v in inputs.items()}
    output = model(**inputs)
    print(f"output: {output}")
    logits = output.logits
    print(f"logits: {logits}")
    pred = torch.argmax(logits, dim=-1)
    print(f"pred: {pred}")
    print(f"输入: {sen}\n模型预测结果:{id2_label.get(pred.item())}")

output: SequenceClassifierOutput(loss=None, logits=tensor([[-2.7260,  2.5605]], device='cuda:0'), hidden_states=None, attentions=None)
logits: tensor([[-2.7260,  2.5605]], device='cuda:0')
pred: tensor([1], device='cuda:0')
输入: 我觉得这家酒店不错，饭很好吃！
模型预测结果:好评！


In [22]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [23]:
pipe(sen)

[{'label': '好评！', 'score': 0.9949659705162048}]